# Data as a matrix: Treasury yields

1. What does it mean for a computer to learn from data?
2. How does a table of yields become numbers a computer can compare?
3. How do a few numbers describe one column, and two columns together?
4. Which kind of problem is a question, and which method answers it?

Daily US Treasury yields for 11 maturities, 9 February 2006 to 30 September 2026, in per cent.

Treasury yields: Federal Reserve Board H.15 via FRED, Federal Reserve Bank of St. Louis. Recession dates: NBER via FRED, Federal Reserve Bank of St. Louis. Policy rates and consumer prices: BIS Data Portal. Snapshot 30 September 2026.

In [ ]:
import pathlib

import pandas as pd

DATA_URL = "https://digital-ai-finance.github.io/data-science-msc/data/"
SNAPSHOT = "2026-09-30"


def load(name):
    """Read a snapshot: "treasury-curve-daily", "us-recession-monthly" or "bis-economies"."""
    local = pathlib.Path("../data/market") / f"{SNAPSHOT}-{name}.csv"
    if local.exists():  # run from material/notebooks/ before the files are published
        return pd.read_csv(local)
    url = f"{DATA_URL}{SNAPSHOT}-{name}.csv"
    try:
        return pd.read_csv(url)
    except Exception as err:
        raise RuntimeError(f"{url} not reachable ({err}). Download the file in a browser, upload "
                           "it in the Colab file panel and read it with pd.read_csv.") from err

In [ ]:
import numpy as np

curve = load("treasury-curve-daily")
curve["date"] = pd.to_datetime(curve["date"])
curve = curve.set_index("date")
print(curve.shape, curve.index[0].date(), curve.index[-1].date())
curve.tail()

## 1. Learning from data

The target is the column a model predicts, the 10-year yield `DGS10`. The features are the columns it reads, the other 10 maturities. A model that copies the nearest training day scores zero error on its training days; only days it never saw measure what it learned.

In [ ]:
TARGET = "DGS10"
FEATURES = [c for c in curve.columns if c != TARGET]
print("target:", TARGET)
print("features:", FEATURES)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

X, y = curve[FEATURES], curve[TARGET]
n = len(curve)
k = int(n * 0.8)
splits = {
    "random": train_test_split(np.arange(n), test_size=0.2, random_state=42),
    "by date": (np.arange(k), np.arange(k, n)),
}
models = [("nearest day", KNeighborsRegressor(n_neighbors=1)), ("linear", LinearRegression())]
for name, model in models:
    for split, (train, test) in splits.items():
        model.fit(X.iloc[train], y.iloc[train])
        tr = mean_squared_error(y.iloc[train], model.predict(X.iloc[train])) ** 0.5
        te = mean_squared_error(y.iloc[test], model.predict(X.iloc[test])) ** 0.5
        print(f"{name:12s} {split:8s} train {tr:.3f} test {te:.3f}")

**Exercise 1.** Use 15 neighbours, `KNeighborsRegressor(n_neighbors=15)`. Which of the four test errors changes most, and why?

## 2. Data as a matrix

Each row of `curve` is one day's yield curve, a vector of 11 yields in per cent. The Euclidean distance between two rows compares two days; a dot product with a weight vector scores every day at once.

In [ ]:
import matplotlib.pyplot as plt

M = curve.to_numpy()
last = M[-1]
dist = pd.Series(np.linalg.norm(M - last, axis=1), index=curve.index)
older = dist[dist.index <= curve.index[-1] - pd.DateOffset(days=90)]
print("nearest day overall:", dist.iloc[:-1].idxmin().date(), round(dist.iloc[:-1].min(), 3))
print("nearest day more than 90 days back:", older.idxmin().date(), round(older.min(), 2))
years = [1 / 12, 0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30]
plt.plot(years, last, "o-", color="#ff7f0e", label=str(curve.index[-1].date()))
plt.plot(years, curve.loc[older.idxmin()], "o-", color="#7f7f7f", label=str(older.idxmin().date()))
plt.xscale("log")
plt.xlabel("maturity, years")
plt.ylabel("yield, per cent")
plt.legend()
plt.show()

In [ ]:
pair = pd.DataFrame({"DGS10": curve["DGS10"], "change_bp": curve["DGS10"].diff() * 100}).dropna()
for label, frame in [("both in per cent", pair.assign(change_bp=pair["change_bp"] / 100)),
                     ("change in basis points", pair),
                     ("standardised", (pair - pair.mean()) / pair.std())]:
    d = np.linalg.norm(frame.to_numpy() - frame.iloc[-1].to_numpy(), axis=1)
    print(label, "nearest days:", list(frame.index[np.argsort(d)[1:4]].date))

In [ ]:
beta = np.zeros(curve.shape[1])
beta[curve.columns.get_loc("DGS3MO")] = -1
beta[curve.columns.get_loc("DGS10")] = 1
spread = M @ beta
print("term spread on the last day:", round(spread[-1], 2))
print("days with the 10-year below the 3-month yield:", int((spread < 0).sum()))

**Exercise 2.** Compute the distance between the first two days with the formula $\sqrt{\sum_j (x_j - y_j)^2}$, and compare with `np.linalg.norm`.

## 3. Describing columns

Mean and median give the centre of a column, the standard deviation its spread, covariance and correlation the joint movement of two columns. Covariance and correlation are dot products of centred vectors.

In [ ]:
bill = curve["DGS3MO"]
print("mean", round(bill.mean(), 2), "median", round(bill.median(), 2), "sd", round(bill.std(), 2))
print("share below the mean", round((bill < bill.mean()).mean() * 100, 1), "%")
print("days below 0.25:", int((bill < 0.25).sum()))
plt.hist(bill, bins=np.arange(0, 6.0, 0.25), color="#7f7f7f")
plt.axvline(bill.mean(), color="#d62728", label="mean")
plt.axvline(bill.median(), color="#0066cc", label="median")
plt.legend()
plt.xlabel("3-month yield, per cent")
plt.show()

In [ ]:
two, ten = curve["DGS2"], curve["DGS10"]
a, b = two - two.mean(), ten - ten.mean()
print(f"covariance: {a @ b / (len(a) - 1):.3f} {two.cov(ten):.3f}")
print(f"r: {(a @ b) / np.sqrt((a @ a) * (b @ b)):.3f} {two.corr(ten):.3f}")
plt.scatter(two, ten, s=2, color="#7f7f7f")
plt.xlabel("2-year yield, per cent")
plt.ylabel("10-year yield, per cent")
plt.show()

In [ ]:
Xc = np.column_stack([a, b])  # the two centred columns as one matrix
print("X_c^T X_c / (n - 1):", (Xc.T @ Xc / (len(Xc) - 1)).round(3).tolist())
print("np.cov:             ", np.cov(two, ten).round(3).tolist())

In [ ]:
zero = bill < 0.25
for label, mask in [("all days", bill == bill), ("3-month below 0.25", zero), ("other days", ~zero)]:
    print(f"{label:20s} {int(mask.sum()):5d} days  r = {bill[mask].corr(ten[mask]):.3f}")

In [ ]:
X = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
quartet = {
    "I": (X, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    "II": (X, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    "III": (X, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    "IV": ([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
           [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}
fig, axes = plt.subplots(2, 2, figsize=(8, 6), sharex=True, sharey=True)
for ax, (name, (x, y)) in zip(axes.flat, quartet.items()):
    ax.scatter(x, y, color="#7f7f7f")
    ax.plot([3, 20], [3 + 0.5 * 3, 3 + 0.5 * 20], color="#d62728")
    ax.set_title(f"{name}: r = {np.corrcoef(x, y)[0, 1]:.2f}")
plt.show()

**Exercise 3.** Compute r between the daily changes of the 2-year and 10-year yields, `curve.diff()`. Compare it with 0.840.

## 4. Problem classes

Three tables: the Treasury curve with its target by choice (deck 9.1 takes the daily change of the 10-year yield, deck 13.1 none), the recession months with the target `recession_in_12m`, the BIS economies with no target.

In [ ]:
rec = load("us-recession-monthly")
econ = load("bis-economies")
print("recession months:", rec.shape, rec["recession_in_12m"].value_counts().to_dict())
print("economies:", econ.shape, list(econ.columns))

**Exercise 4.** Sort five questions into regression, classification, clustering and dimensionality reduction:

1. tomorrow's change of the 10-year yield;
2. whether there is a recession month 12 months later;
3. groups of economies by policy rate and inflation;
4. three axes for 11 maturities;
5. whether the 10-year yield stands higher in a month.

## Newer data (optional)

The cells below fetch the live series from FRED. Their numbers differ from the slides, which use the snapshot of 30 September 2026.

In [ ]:
FRED = "https://fred.stlouisfed.org/graph/fredgraph.csv?id="
BIS = "https://stats.bis.org/api/v1/data/{}?startPeriod={}&format=csv"


def fred(*ids):
    """Live FRED series, e.g. fred("GS10", "TB3MS", "USREC"); "." marks a missing value."""
    return pd.read_csv(FRED + ",".join(ids), na_values=["."], parse_dates=["observation_date"])


def bis(flow, start):
    """Live BIS series, e.g. bis("WS_CBPOL/M..", "2021-12")."""
    return pd.read_csv(BIS.format(flow, start))


live = fred(*curve.columns).dropna()
print(live.shape, live["observation_date"].iloc[-1].date())